# Phase 2 — Multi-Seed, Multi-Encoder Training Matrix

Chapter 4 currently reports a **single run** of a single encoder. A single run gives no
handle on seed variance, so it cannot support a claim that one encoder beats another. This
notebook replaces the point estimates with mean +/- sd over three seeds per configuration
and settles the encoder question on our own data.

**The encoder question.** A panelist's published work reports XLNet outperforming BERT on
privacy-policy classification; the thesis claims legal-domain pretraining dominates. Both
claims are about *other* corpora. Running all four encoders through an identical protocol
on identical splits is the only way to know which holds here.

## Run matrix

| Axis | Values |
| --- | --- |
| Encoders (dual-head) | `nlpaueb/legal-bert-base-uncased`, `bert-base-uncased`, `xlnet-base-cased`, `roberta-base` |
| Seeds | 42, 1337, 2024 |
| Head ablation (**legal-bert only**) | topic-only, risk-only (dual-head reuses the runs above) |

**12 encoder runs + 6 ablation runs = 18.** The head ablation is deliberately *not* run for
the other three encoders: it answers "does joint training help?", which is a question about
the architecture, not about the backbone, and 18 runs is already the practical ceiling.

## What is held fixed

Everything except encoder / seed / head mode, and it is held fixed by construction — the
runner imports `scripts/lawgic_train_matrix.py`, which reads the same persisted seed-42
split file, the same taxonomy, the same masked-BCE + masked-CE losses (copied line for
line from the original `DualHeadTrainer`), lr 3e-5, batch 8, up to 20 epochs, early
stopping patience 3, weight decay 0.01, warmup 0.06, FP16 on CUDA, max_length 256, and the
same pre-training degenerate-model assertion (a zero-logit model must score topic macro-F1
below 0.95).

## How the pooled representation is chosen per architecture

The two linear heads read one vector per clause. Which token that vector comes from is
**not** the same across these four encoders, and getting it wrong silently cripples a
model rather than erroring:

- **BERT, Legal-BERT, RoBERTa** — the sequence summary is the **first** token
  (`[CLS]` / `<s>`), placed there during pretraining.
- **XLNet** — XLNet is trained with the summary token **appended at the end**. Reading
  position 0 would hand the head an ordinary content token. So XLNet uses the **last**
  token.

`pooled_representation()` in `scripts/lawgic_train_matrix.py` is the single place this
lives. It selects by attention mask rather than by fixed index (`attention_mask.argmax(1)`
for first, `L - 1 - flip(mask).argmax(1)` for last), because XLNet's tokenizer pads on the
**left** while the BERT-family tokenizers pad on the right — a hardcoded `[:, 0]` or
`[:, -1]` would read padding for one of them.

Two further per-architecture quirks are handled in the same adapter, not scattered around:

- **RoBERTa has no `token_type_ids`.** The collator keeps only the keys in
  `tokenizer.model_input_names`, so each tokenizer declares its own contract and no
  `if roberta:` branch is needed anywhere.
- **XLNet's tokenizer needs `sentencepiece`.** Already present in
  `notebooks/requirements.txt` (`sentencepiece==0.2.1`); listed as a manual check below.

**Deviation to record in the manuscript.** The original v3 checkpoint fed the heads BERT's
`pooler_output` (a dense+tanh layer on top of `[CLS]`). The matrix uses the raw first
token instead, for all encoders. Reason: `roberta-base` ships with a *randomly initialised*
pooler, so keeping `pooler_output` would have handicapped RoBERTa for reasons unrelated to
the encoder itself. Consistency across the four arms matters more than bit-matching the
old run, so the legal-bert/seed-42 cell of this matrix is **not** expected to reproduce the
v3 checkpoint exactly — treat the matrix as internally comparable and the Phase 1 numbers
as the checkpoint's own.

In [1]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    sentinel = Path("generated_files/lawgic_taxonomy/lawgic_multihead_wide.csv")
    for candidate in (start, *start.parents):
        if (candidate / sentinel).exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the lawgic repository.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

import json

import numpy as np
import pandas as pd

import lawgic_eval_core as core
import lawgic_train_matrix as tm

pd.set_option("display.width", 160)

split_path = core.persist_split()
corpus = core.load_corpus()
frames = core.split_frames(corpus)
assert {k: len(v) for k, v in frames.items()} == core.EXPECTED_SPLIT_ROWS
print(f"Split artifact: {split_path}")
print("Rows:", {k: len(v) for k, v in frames.items()})

MATRIX = tm.build_matrix()
print(f"\nConfigured runs: {len(MATRIX)}")
display(pd.DataFrame([{
    "run_id": c.run_id, "encoder": c.encoder_name, "seed": c.seed,
    "heads": c.heads, "selection_metric": c.best_metric_key,
} for c in MATRIX]))

Split artifact: C:\Users\Enrique\Coding Projects\Thesis\lawgic\generated_files\lawgic_taxonomy\splits\split_seed42.csv
Rows: {'train': 21183, 'validation': 2648, 'test': 2648}

Configured runs: 18


,run_id,encoder,seed,heads,selection_metric
0,legal-bert-base-uncased__seed42__dual,nlpaueb/legal-bert-base-uncased,42,dual,topic_macro_f1
1,legal-bert-base-uncased__seed1337__dual,nlpaueb/legal-bert-base-uncased,1337,dual,topic_macro_f1
2,legal-bert-base-uncased__seed2024__dual,nlpaueb/legal-bert-base-uncased,2024,dual,topic_macro_f1
3,bert-base-uncased__seed42__dual,bert-base-uncased,42,dual,topic_macro_f1
4,bert-base-uncased__seed1337__dual,bert-base-uncased,1337,dual,topic_macro_f1
5,bert-base-uncased__seed2024__dual,bert-base-uncased,2024,dual,topic_macro_f1
6,xlnet-base-cased__seed42__dual,xlnet-base-cased,42,dual,topic_macro_f1
7,xlnet-base-cased__seed1337__dual,xlnet-base-cased,1337,dual,topic_macro_f1
8,xlnet-base-cased__seed2024__dual,xlnet-base-cased,2024,dual,topic_macro_f1
9,roberta-base__seed42__dual,roberta-base,42,dual,topic_macro_f1


### Adding two extra legal-bert seeds

The matrix is a list of `RunConfig` dataclasses, so extending it is one line. Uncomment
below if you want five seeds for legal-bert (dual-head only) rather than three. Do **not**
add seeds to only some arms and then compare sds across arms — the sd of 5 draws is not
comparable to the sd of 3.

In [ ]:
# MATRIX += [tm.RunConfig(encoder_name=tm.ENCODERS[0], seed=s, heads="dual") for s in (7, 2718)]
# print(f"Runs after extension: {len(MATRIX)}")

## MANUAL STEP — before running the matrix

1. **Model downloads.** The first run of each encoder pulls weights from the HuggingFace
   hub (~440 MB each for `bert-base-uncased`, `xlnet-base-cased`, `roberta-base`;
   legal-bert is already local). Requires network access on the training machine. The
   cell below pre-fetches all three in-notebook via `AutoModel`/`AutoTokenizer`.
2. **`sentencepiece`** must be importable for the XLNet tokenizer. It is already in
   `notebooks/requirements.txt`; the check cell below verifies it rather than installing it.
3. **GPU.** These are 18 full fine-tunes. On CPU this is days, not hours — run on the CUDA
   machine that produced the v3 checkpoint. FP16 switches on automatically on CUDA and off
   elsewhere, matching the original protocol.
4. **Disk.** Each run keeps one checkpoint (`save_total_limit=1`), ~440 MB, plus a small
   `test_logits.npz`. Budget ~10 GB for the full matrix under
   `generated_files/lawgic_taxonomy/runs/`.

Nothing here writes to `saved_models/`; the deployed v3 checkpoint is never touched.

In [5]:
from transformers import AutoModel, AutoTokenizer

MODELS_TO_PREFETCH = ["bert-base-uncased", "xlnet-base-cased", "roberta-base"]

for model_name in MODELS_TO_PREFETCH:
    print(f"Downloading {model_name} ...")
    AutoTokenizer.from_pretrained(model_name)
    AutoModel.from_pretrained(model_name)
    print(f"  done: {model_name}")

print("\nAll three encoders cached locally.")

  done: bert-base-uncased
  done: xlnet-base-cased


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  done: roberta-base

All three encoders cached locally.


In [9]:
%conda install conda-forge::sentencepiece

3 channel Terms of Service accepted
Channels:
 - defaults
 - conda-forge
Platform: win-64
Solving environment: done

## Package Plan ##

  environment location: c:\Users\Enrique\anaconda3\envs\thesis-env

  added / updated specs:
    - conda-forge::sentencepiece


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    libabseil-20260526.0       | cxx17_h4cdcee1_0         1.9 MB
    libprotobuf-7.35.1         |       hb5abd84_0         6.9 MB
    libsentencepiece-0.2.1     |       h1e80020_4         1.4 MB  conda-forge
    sentencepiece-0.2.1        |       hb9477dd_4          20 KB  conda-forge
    sentencepiece-python-0.2.1 |  py314h2f88111_4         3.2 MB  conda-forge
    sentencepiece-spm-0.2.1    |       h1e80020_4         173 KB  conda-forge
    ------------------------------------------------------------
                                           Total:        13.5 MB

The following NEW 



==> WARNING: A newer version of conda exists. <==
    current version: 25.5.1
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c defaults conda




In [10]:
import importlib.util

print("sentencepiece:", "OK" if importlib.util.find_spec("sentencepiece") else "MISSING — XLNet will fail")
print("scipy:", "OK" if importlib.util.find_spec("scipy") else "MISSING — McNemar will fail")

device_label, device = tm.detect_device()
print(f"device: {device_label} (fp16={device_label == 'cuda'})")
if device_label != "cuda":
    print("WARNING: not on CUDA. The matrix will take days. Stop and move to the GPU machine.")

sentencepiece: OK
scipy: OK
device: cuda (fp16=True)


## Expected wall time

The v3 run's `trainer_state.json` records **17 epochs** before early stopping (best at
epoch 14, patience 3) at batch size 8 over 21,183 training rows = 2,648 optimizer steps
per epoch, and an eval throughput of ~455 clauses/s on the original CUDA device. It does
**not** record `train_runtime` — the notebook that produced it never logged the summary —
so the per-run wall time must be **measured on the first run**, not assumed.

The cell below prints the derived lower bound from what *is* recorded, then the runner
stores the real `wall_seconds` for every run. After the first run completes, multiply.

In [11]:
state_path = core.CHECKPOINT_DIR / "checkpoints/checkpoint-45016/trainer_state.json"
if state_path.exists():
    state = json.loads(state_path.read_text())
    evals = [h for h in state["log_history"] if "eval_runtime" in h]
    eval_throughput = float(np.mean([h["eval_samples_per_second"] for h in evals]))
    epochs = float(state["epoch"])
    # Training is roughly 3-4x the cost of inference per sample (forward + backward + optimizer).
    optimistic_seconds = epochs * (len(frames["train"]) / (eval_throughput / 3.5))
    print(f"v3 run: {epochs:.0f} epochs, eval throughput {eval_throughput:.0f} clauses/s")
    print(f"Derived LOWER BOUND per run: ~{optimistic_seconds / 60:.0f} min "
          f"-> ~{18 * optimistic_seconds / 3600:.1f} h for 18 runs")
    print("This is an extrapolation, not a measurement. Trust wall_seconds from run 1 instead.")
else:
    print("No v3 trainer_state.json found; wall time must be measured on the first run.")

v3 run: 17 epochs, eval throughput 445 clauses/s
Derived LOWER BOUND per run: ~47 min -> ~14.2 h for 18 runs
This is an extrapolation, not a measurement. Trust wall_seconds from run 1 instead.


## Runner

Each config trains, evaluates on the frozen test split, and writes to
`generated_files/lawgic_taxonomy/runs/<run_id>/`:

- `metrics.json` — config + headline test metrics + `wall_seconds` + `epochs_run`
- `test_logits.npz` — test logits, labels and masks (so aggregation, bootstrap and paired
  tests never need to re-run inference)
- `per_topic.csv` — per-topic precision / recall / F1 / support

Completed runs are skipped, so the cell is **resumable**: interrupt it, restart the kernel,
re-run. Set `FORCE_RERUN = True` to redo everything.

In [ ]:
FORCE_RERUN = False

records = []
for index, config in enumerate(MATRIX, start=1):
    target = tm.RUNS_DIR / config.run_id / "metrics.json"
    if target.exists() and not FORCE_RERUN:
        print(f"[{index}/{len(MATRIX)}] skip {config.run_id} (already complete)")
        records.append(json.loads(target.read_text()))
        continue
    print(f"[{index}/{len(MATRIX)}] running {config.run_id} ...")
    records.append(tm.run_config(config))

print(f"\nCompleted {len(records)} runs.")

[1/18] running legal-bert-base-uncased__seed42__dual ...


You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Topic Macro F1,Topic Micro F1,Topic Weighted F1,Topic Observed Positions,Risk Accuracy,Risk Macro F1,Risk Weighted F1,Risk Valid Rows
1,0.769300,0.775791,0.064763,0.211135,0.171681,96620.000000,0.804003,0.797135,0.803743,2648.000000
2,0.604600,0.690023,0.381158,0.631470,0.574289,96620.000000,0.814955,0.805029,0.814502,2648.000000
3,0.375200,0.640507,0.603367,0.755805,0.733281,96620.000000,0.837991,0.832059,0.836521,2648.000000
4,0.388900,0.719763,0.642261,0.782893,0.770978,96620.000000,0.841390,0.835409,0.839652,2648.000000
5,0.249500,0.825951,0.669976,0.803822,0.795191,96620.000000,0.843656,0.839919,0.844028,2648.000000
6,0.146400,0.876684,0.704711,0.824830,0.818666,96620.000000,0.848565,0.843226,0.848650,2648.000000
7,0.138400,1.025527,0.710526,0.829706,0.825624,96620.000000,0.853097,0.846846,0.852055,2648.000000
8,0.092200,1.074190,0.732378,0.832284,0.829008,96620.000000,0.845544,0.840759,0.845042,2648.000000
9,0.046300,1.102909,0.744242,0.838182,0.835651,96620.000000,0.848565,0.843589,0.848294,2648.000000
10,0.095000,1.162956,0.739702,0.837367,0.835221,96620.000000,0.847810,0.842722,0.847207,2648.000000


[legal-bert-base-uncased__seed42__dual] 63.8 min | topic_macro_f1=0.7741 topic_micro_f1=0.8363 risk_accuracy=0.8406 risk_macro_f1=0.8349
[2/18] running legal-bert-base-uncased__seed1337__dual ...


You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Topic Macro F1,Topic Micro F1,Topic Weighted F1,Topic Observed Positions,Risk Accuracy,Risk Macro F1,Risk Weighted F1,Risk Valid Rows
1,0.665100,0.841575,0.098436,0.292189,0.237333,96620.000000,0.786631,0.776126,0.786774,2648.000000
2,0.571300,0.659903,0.437594,0.649453,0.592348,96620.000000,0.816843,0.813411,0.817000,2648.000000
3,0.442800,0.704480,0.572639,0.737571,0.709921,96620.000000,0.835725,0.828719,0.835259,2648.000000
4,0.263400,0.731843,0.650250,0.790295,0.777764,96620.000000,0.841767,0.835669,0.841540,2648.000000
5,0.275800,0.765842,0.668034,0.804688,0.792548,96620.000000,0.839502,0.832701,0.838285,2648.000000
6,0.287000,0.880332,0.689551,0.820658,0.815068,96620.000000,0.847432,0.842175,0.846934,2648.000000
7,0.183700,1.001644,0.729665,0.829157,0.825064,96620.000000,0.845921,0.844195,0.846698,2648.000000
8,0.069500,1.059433,0.732081,0.836083,0.831635,96620.000000,0.853474,0.850188,0.853387,2648.000000
9,0.061700,1.115113,0.770266,0.835931,0.833880,96620.000000,0.858006,0.854155,0.858445,2648.000000
10,0.089100,1.186666,0.756061,0.836447,0.834530,96620.000000,0.852341,0.847797,0.852053,2648.000000


[legal-bert-base-uncased__seed1337__dual] 45.3 min | topic_macro_f1=0.7693 topic_micro_f1=0.8329 risk_accuracy=0.8380 risk_macro_f1=0.8329
[3/18] running legal-bert-base-uncased__seed2024__dual ...


You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Topic Macro F1,Topic Micro F1,Topic Weighted F1,Topic Observed Positions,Risk Accuracy,Risk Macro F1,Risk Weighted F1,Risk Valid Rows
1,0.707900,0.842565,0.091213,0.295473,0.229743,96620.000000,0.786631,0.772659,0.784099,2648.000000
2,0.555800,0.630114,0.416257,0.646568,0.586441,96620.000000,0.823640,0.818654,0.824256,2648.000000
3,0.369000,0.668306,0.596066,0.758979,0.735614,96620.000000,0.840257,0.833411,0.839364,2648.000000


In [ ]:
# Save the best (highest validation metric) dual-head model per encoder to
# saved_models/, in the same file layout as lawgic_classifier_legal-bert_v3
# (model_state_dict.pt + encoder/tokenizer + head weights + taxonomy + metadata).
# Reads metrics.json directly from disk, so it works even for encoders whose
# runs finished before the rest of the matrix does. Writes to a NEW directory
# per encoder (suffix "_phase2") — never touches lawgic_classifier_legal-bert_v3.
# Skips an encoder whose target directory already exists (idempotent / resumable),
# and skips an encoder with no completed dual-head run yet.

import shutil
from datetime import datetime, timezone

import torch
from safetensors.torch import load_file as load_safetensors
from transformers import AutoTokenizer

SAVE_TARGETS = {
    "nlpaueb/legal-bert-base-uncased": "legal-bert",
    "bert-base-uncased": "bert",
    "xlnet-base-cased": "xlnet",
    "roberta-base": "roberta",
}
SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models"


def completed_dual_runs(encoder_name: str) -> list[dict]:
    records = []
    for metrics_path in sorted(tm.RUNS_DIR.glob("*/metrics.json")):
        record = json.loads(metrics_path.read_text())
        if record["encoder_name"] == encoder_name and record["heads"] == "dual":
            records.append(record)
    return records


def best_checkpoint_dir(run_id: str) -> Path:
    checkpoints = sorted(
        (tm.RUNS_DIR / run_id / "checkpoints").glob("checkpoint-*"),
        key=lambda p: int(p.name.split("-")[-1]),
    )
    if not checkpoints:
        raise FileNotFoundError(f"No checkpoint saved for {run_id}")
    # save_total_limit=1 + load_best_model_at_end=True: the one surviving
    # checkpoint is the best validation checkpoint, not just the last epoch.
    return checkpoints[-1]


def save_best_model(encoder_name: str, short_name: str) -> None:
    candidates = completed_dual_runs(encoder_name)
    if not candidates:
        print(f"skip {short_name}: no completed dual-head runs yet")
        return

    best = max(candidates, key=lambda r: r["best_val_metric"])
    run_id = best["run_id"]

    output_dir = SAVED_MODELS_DIR / f"lawgic_classifier_{short_name}_phase2"
    if output_dir.exists():
        print(f"skip {short_name}: {output_dir} already exists, not overwriting")
        return

    checkpoint_dir = best_checkpoint_dir(run_id)

    model = tm.LawgicDualHeadModel(encoder_name)
    weights_file = checkpoint_dir / "model.safetensors"
    state_dict = (
        load_safetensors(str(weights_file))
        if weights_file.exists()
        else torch.load(checkpoint_dir / "pytorch_model.bin", map_location="cpu", weights_only=True)
    )
    model.load_state_dict(state_dict)

    tokenizer = AutoTokenizer.from_pretrained(str(checkpoint_dir))

    output_dir.mkdir(parents=True)

    # Full state dict + encoder/tokenizer + heads separately, mirroring v3's layout.
    torch.save(model.state_dict(), output_dir / "model_state_dict.pt")
    model.encoder.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    torch.save(model.topic_head.state_dict(), output_dir / "topic_head_weights.pt")
    torch.save(model.harm_head.state_dict(), output_dir / "harm_head_weights.pt")

    topic_ids, name_by_topic, _ = core.load_taxonomy()
    compact_taxonomy = [
        {"classifier_id": i, "topic_id": tid, "name": name_by_topic[tid]}
        for i, tid in enumerate(topic_ids)
    ]
    (output_dir / "lawgic_topics_44.json").write_text(json.dumps(compact_taxonomy, indent=2))
    shutil.copy2(core.TAXONOMY_PATH, output_dir / "lawgic_topics_original_45.json")

    (output_dir / "test_metrics.json").write_text(json.dumps(best, indent=2, default=str))

    metadata = {
        "model_name": encoder_name,
        "architecture": "dual_head",
        "num_topics": core.NUM_LAWGIC_TOPICS,
        "num_harm_classes": core.NUM_HARM_CLASSES,
        "max_length": core.MAX_LENGTH,
        "decision_threshold": core.DECISION_THRESHOLD,
        "seed": best["seed"],
        "source_run_id": run_id,
        "best_val_metric": best["best_val_metric"],
        "seeds_considered": sorted(r["seed"] for r in candidates),
        "saved_at": datetime.now(timezone.utc).isoformat(),
        "note": (
            "Best-of-3-seeds model from the Phase 2 multi-encoder matrix "
            "(notebooks/evaluation/02_multiseed_encoder_runs.ipynb); does not "
            "replace lawgic_classifier_legal-bert_v3."
        ),
    }
    (output_dir / "training_metadata.json").write_text(json.dumps(metadata, indent=2))

    print(f"[{short_name}] saved best seed {best['seed']} (run {run_id}) -> {output_dir}")


for encoder_name, short_name in SAVE_TARGETS.items():
    save_best_model(encoder_name, short_name)

## Aggregation

Everything below reads the persisted run artifacts, so it can be re-run without a GPU.

In [ ]:
run_files = sorted(tm.RUNS_DIR.glob("*/metrics.json"))
runs = pd.DataFrame([json.loads(p.read_text()) for p in run_files])
runs = runs[runs["holdout_source"].isna()] if "holdout_source" in runs else runs
print(f"Loaded {len(runs)} Phase 2 runs from {tm.RUNS_DIR}")
display(runs[["run_id", "encoder_name", "seed", "heads", "epochs_run", "wall_seconds", *core.HEADLINE_METRICS]])

runs.to_csv(core.EVAL_OUT_DIR / "phase2_runs.csv", index=False)
print(f"\nMeasured wall time: {runs['wall_seconds'].mean() / 60:.1f} min/run "
      f"(total {runs['wall_seconds'].sum() / 3600:.1f} h)")

In [ ]:
grouped = runs.groupby(["encoder_name", "heads"])
aggregate = grouped[list(core.HEADLINE_METRICS)].agg(["mean", "std", "count"])
aggregate.columns = ["_".join(c) for c in aggregate.columns]
aggregate = aggregate.reset_index()
display(aggregate)
aggregate.to_csv(core.EVAL_OUT_DIR / "phase2_aggregate.csv", index=False)

### Bootstrap CIs on the test metrics

Per run, 1,000 clause-level resamples of the test split, computed from the stored logits.
Reported alongside the across-seed sd: the bootstrap CI measures *test-set* sampling
noise, the sd measures *initialisation/ordering* noise. They are different quantities and
the manuscript should not conflate them.

In [ ]:
N_RESAMPLES = 1000


def load_run_logits(run_id: str) -> dict:
    payload = np.load(tm.RUNS_DIR / run_id / "test_logits.npz")
    return {
        "topic_logits": payload["topic_logits"],
        "harm_logits": payload["harm_logits"],
        "arrays": {
            "labels": payload["labels"],
            "label_masks": payload["label_masks"],
            "harm_labels": payload["harm_labels"],
            "harm_masks": payload["harm_masks"],
        },
        "row_id": payload["row_id"],
    }


ci_rows = []
for run_id in runs["run_id"]:
    payload = load_run_logits(run_id)
    ci = core.bootstrap_ci(
        payload["topic_logits"], payload["harm_logits"], payload["arrays"], n_resamples=N_RESAMPLES
    )
    ci.insert(0, "run_id", run_id)
    ci_rows.append(ci)

bootstrap_table = pd.concat(ci_rows, ignore_index=True)
bootstrap_table.to_csv(core.EVAL_OUT_DIR / "phase2_bootstrap_ci.csv", index=False)
display(bootstrap_table.head(12))

### Paired significance tests

Both tests are **paired on the clause**: every run scored the identical test rows in the
identical order, so a difference is attributable to the varied component and nothing else.

- **Risk head — McNemar.** Item-level correctness per clause (over `harm_mask=1` rows),
  exact binomial on the discordant pairs. This is the right test for two classifiers on
  one sample; an unpaired accuracy comparison would throw away the pairing and lose power.
- **Topic head — paired bootstrap.** Macro-F1 is not decomposable into per-item
  correctness, so McNemar does not apply. Instead each resample draws one set of clause
  indices and scores *both* models on it; the reported interval is over the difference.

Seeds are averaged out by comparing the **best seed** of each arm; change `pick` below to
compare a fixed seed if you would rather not condition on validation performance.

In [ ]:
def best_run(encoder: str, heads: str = "dual") -> str:
    subset = runs[(runs["encoder_name"] == encoder) & (runs["heads"] == heads)]
    if subset.empty:
        raise KeyError(f"no runs for {encoder}/{heads}")
    return subset.sort_values("best_val_metric", ascending=False).iloc[0]["run_id"]


def compare(run_a: str, run_b: str) -> dict:
    a, b = load_run_logits(run_a), load_run_logits(run_b)
    assert np.array_equal(a["row_id"], b["row_id"]), "runs were scored on different rows"
    arrays = a["arrays"]

    valid = arrays["harm_masks"].astype(bool)
    correct_a = a["harm_logits"][valid].argmax(1) == arrays["harm_labels"][valid]
    correct_b = b["harm_logits"][valid].argmax(1) == arrays["harm_labels"][valid]
    mcnemar = core.mcnemar(correct_a, correct_b)

    def delta(indices):
        ma = core.topic_metrics(a["topic_logits"][indices], arrays["labels"][indices], arrays["label_masks"][indices])
        mb = core.topic_metrics(b["topic_logits"][indices], arrays["labels"][indices], arrays["label_masks"][indices])
        return ma["topic_macro_f1"] - mb["topic_macro_f1"]

    paired = core.paired_bootstrap_delta(delta, np.arange(len(arrays["labels"])), n_resamples=N_RESAMPLES)
    return {
        "run_a": run_a,
        "run_b": run_b,
        "risk_mcnemar_b": mcnemar["b"],
        "risk_mcnemar_c": mcnemar["c"],
        "risk_mcnemar_p": mcnemar["p_value"],
        "topic_macro_f1_delta": paired["delta"],
        "topic_delta_ci_low": paired["ci_low"],
        "topic_delta_ci_high": paired["ci_high"],
        "topic_delta_p": paired["p_value"],
    }


LEGAL_BERT = tm.ENCODERS[0]
comparisons = []
for other in tm.ENCODERS[1:]:
    try:
        comparisons.append(compare(best_run(LEGAL_BERT), best_run(other)))
    except KeyError as exc:
        print(f"skipped: {exc}")

# Head ablation: dual vs each single-head variant, legal-bert only.
for heads in ("topic", "risk"):
    try:
        comparisons.append(compare(best_run(LEGAL_BERT), best_run(LEGAL_BERT, heads)))
    except KeyError as exc:
        print(f"skipped: {exc}")

significance = pd.DataFrame(comparisons)
significance.to_csv(core.EVAL_OUT_DIR / "phase2_significance.csv", index=False)
display(significance)

## Output table (a) — headline, rows = encoder/config

Cells are `mean +/- sd` over seeds. `n/a` marks a metric a configuration cannot produce:
topic-only leaves the risk head untrained, risk-only leaves the topic head untrained, so
reporting those cells would be reporting random weights.

In [ ]:
LABELS = {
    "topic_macro_f1": "Topic macro-F1",
    "topic_micro_f1": "Topic micro-F1",
    "risk_accuracy": "Risk accuracy",
    "risk_macro_f1": "Risk macro-F1",
}
CONFIG_NAMES = {
    ("nlpaueb/legal-bert-base-uncased", "dual"): "Legal-BERT (dual)",
    ("bert-base-uncased", "dual"): "BERT (dual)",
    ("xlnet-base-cased", "dual"): "XLNet (dual)",
    ("roberta-base", "dual"): "RoBERTa (dual)",
    ("nlpaueb/legal-bert-base-uncased", "topic"): "Legal-BERT (topic-only)",
    ("nlpaueb/legal-bert-base-uncased", "risk"): "Legal-BERT (risk-only)",
}


def mean_sd(values: pd.Series) -> str:
    if values.isna().all():
        return "n/a"
    return f"{values.mean():.3f} ± {values.std(ddof=1):.3f}" if len(values) > 1 else f"{values.mean():.3f}"


headline = pd.DataFrame(
    [
        {
            "Configuration": CONFIG_NAMES.get((encoder, heads), f"{encoder} ({heads})"),
            "Seeds": int(group["seed"].nunique()),
            **{LABELS[m]: mean_sd(group[m]) for m in core.HEADLINE_METRICS},
        }
        for (encoder, heads), group in runs.groupby(["encoder_name", "heads"])
    ]
)
order = [CONFIG_NAMES[k] for k in CONFIG_NAMES if CONFIG_NAMES[k] in set(headline["Configuration"])]
headline = headline.set_index("Configuration").loc[order].reset_index()
display(headline)

core.write_outputs(
    headline,
    "phase2_headline",
    caption=(
        "Test performance by encoder and head configuration, mean $\\pm$ standard deviation "
        "over three seeds (42, 1337, 2024). All runs use the identical persisted seed-42 "
        "clause split and identical hyperparameters; only the encoder, the seed and the "
        "active heads vary."
    ),
    label="tab:encoder-matrix",
)

## Output table (b) — per-topic breakdown for the final model

Rows = 44 topics + macro avg + weighted avg, columns = precision / recall / F1 / support.
Reported twice: for the **best legal-bert seed** (what a deployed single model achieves)
and as the **mean across the three seeds** (what the architecture achieves). Support is
identical across seeds because the split is.

In [ ]:
topic_ids, name_by_topic, _ = core.load_taxonomy()

best_legal_bert = best_run(LEGAL_BERT)
best_table = pd.read_csv(tm.RUNS_DIR / best_legal_bert / "per_topic.csv")

seed_tables = [
    pd.read_csv(tm.RUNS_DIR / run_id / "per_topic.csv").set_index("topic_id")
    for run_id in runs[(runs["encoder_name"] == LEGAL_BERT) & (runs["heads"] == "dual")]["run_id"]
]
mean_table = sum(t[["precision", "recall", "f1"]] for t in seed_tables) / len(seed_tables)
mean_table = mean_table.join(seed_tables[0][["support", "observed"]]).reset_index()

per_topic = best_table.merge(mean_table, on="topic_id", suffixes=("_best", "_mean"))
per_topic.insert(1, "topic_name", per_topic["topic_id"].map(lambda t: name_by_topic.get(t, t)))
display(per_topic)

core.write_outputs(
    per_topic[["topic_id", "topic_name", "precision_best", "recall_best", "f1_best",
               "f1_mean", "support_best"]].rename(columns={
        "topic_id": "Topic", "topic_name": "Name", "precision_best": "P", "recall_best": "R",
        "f1_best": "F1", "f1_mean": "F1 (seed mean)", "support_best": "Support"}),
    "phase2_per_topic",
    caption=(
        f"Per-topic test performance of the best Legal-BERT dual-head seed ({best_legal_bert}), "
        "with the mean F1 across the three seeds for comparison. Support counts supervised "
        "positive cells in the test split; topics with no observed test cells are omitted."
    ),
    label="tab:per-topic",
)